<a href="https://colab.research.google.com/github/Botatopiece9/kpneumo-amr-prediction/blob/main/02_get_genomes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -q -p /content/amrenv -c conda-forge -c bioconda ncbi-amrfinderplus

bin/micromamba


In [2]:
import os
os.environ["PATH"] = "/content/amrenv/bin:" + os.environ["PATH"]

!amrfinder --version
!amrfinder -u

4.2.7
Running: amrfinder -u
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /content/amrenv/bin/
Software version: 4.2.7
Reverting to hard coded directory: /content/amrenv/share/amrfinderplus/data/latest
Running: /content/amrenv/bin/amrfinder_update -d /content/amrenv/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /content/amrenv/share/amrfinderplus/data/2026-08-07.1/
Running: /content/amrenv/bin/amrfinder_index /content/amrenv/share/amrfinderplus/data/2026-08-07.1/
Indexing
amrfinder_index took 7 seconds to complete
amrfinder_update took 13 seconds to complete
Database directory: /content/amrenv/share/amrfinderplus/data/2026-08-07.1
Database version: 2026-08-07.1
amrfinder took 13 seconds to complete


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import requests
import pandas as pd
import time
import os

PROJECT_DIR = "/content/drive/MyDrive/kpneumo-amr-prediction"

labels = pd.read_csv(f"{PROJECT_DIR}/meropenem_labels_clean.csv", dtype={"genome_id": str})
print(labels.shape)

Mounted at /content/drive
(5820, 3)


In [4]:
SEQ_URL = "https://www.bv-brc.org/api/genome_sequence/"

test_id = labels.loc[labels["label"] == "Resistant", "genome_id"].iloc[0]
print("Test genome:", test_id)

query = f"eq(genome_id,{test_id})&limit(10000)"
response = requests.get(SEQ_URL + "?" + query, headers={"Accept": "application/dna+fasta"})
response.raise_for_status()

with open("test_genome.fasta", "w") as f:
    f.write(response.text)

print(response.text[:300])
print("Number of contigs:", response.text.count(">"))

Test genome: 573.20061
>accn|CP032174   CP032174.1   [Klebsiella pneumoniae strain AR_0160 | 573.20061]
aagcaattagagatcaacatcccggccgatatcctgcgtcgtttcagacagcttccgcaa
ggggtttatccggacggaagtggtactgcattcctcggtatttttaaaacaaaacgtggt
cttatttgccacattctgaaaaacacattaggtccaaaactgctttgggctttgaactca
tctgcaaaagacagagcgctcagagatgtgctttat
Number of contigs: 3


In [5]:
!amrfinder -n test_genome.fasta --organism Klebsiella_pneumoniae --name {test_id} --threads 2 -o test_result.tsv

Running: amrfinder -n test_genome.fasta --organism Klebsiella_pneumoniae --name 573.20061 --threads 2 -o test_result.tsv
Software directory: /content/amrenv/bin/
Software version: 4.2.7
Reverting to hard coded directory: /content/amrenv/share/amrfinderplus/data/latest
Database directory: /content/amrenv/share/amrfinderplus/data/2026-08-07.1
Database version: 2026-08-07.1
AMRFinder translated nucleotide and mutation search
Running tblastn
Running blastn
Making report
amrfinder took 442 seconds to complete


In [6]:
result = pd.read_csv("test_result.tsv", sep="\t")
print(result.columns.tolist())
print(result[["Element symbol", "Class", "Subclass", "Method"]].to_string())

['Name', 'Protein id', 'Contig id', 'Start', 'Stop', 'Strand', 'Element symbol', 'Element name', 'Scope', 'Type', 'Subtype', 'Class', 'Subclass', 'Method', 'Target length', 'Reference sequence length', '% Coverage of reference', '% Identity to reference', 'Alignment length', 'Closest reference accession', 'Closest reference name', 'HMM accession', 'HMM description']
  Element symbol                                       Class                                       Subclass   Method
0      blaOXA-48                                 BETA-LACTAM                                     CARBAPENEM  ALLELEX
1           oqxA  NITROFURAN/PHENICOL/QUINOLONE/TETRACYCLINE  NITROFURANTOIN/PHENICOL/QUINOLONE/TIGECYCLINE   BLASTX
2          oqxB5                          PHENICOL/QUINOLONE                             PHENICOL/QUINOLONE  ALLELEX
3          fosA5                                  FOSFOMYCIN                                     FOSFOMYCIN  ALLELEX
4      blaSHV-11                              

In [7]:
GENOME_URL = "https://www.bv-brc.org/api/genome/"
fields = "genome_id,genome_quality,contigs,genome_length,checkm_completeness,checkm_contamination,mlst"

ids = labels["genome_id"].tolist()
BATCH = 200
meta_records = []

for start in range(0, len(ids), BATCH):
    batch_ids = ids[start:start + BATCH]
    query = f"in(genome_id,({','.join(batch_ids)}))&select({fields})&limit({BATCH})"
    response = requests.get(GENOME_URL + "?" + query, headers={"Accept": "application/json"})
    response.raise_for_status()
    meta_records.extend(response.json())
    print(f"Done {start + len(batch_ids)} / {len(ids)}")
    time.sleep(1)

meta = pd.DataFrame(meta_records)
meta.to_csv(f"{PROJECT_DIR}/genome_metadata_raw.csv", index=False)
print(meta.shape)

Done 200 / 5820
Done 400 / 5820
Done 600 / 5820
Done 800 / 5820
Done 1000 / 5820
Done 1200 / 5820
Done 1400 / 5820
Done 1600 / 5820
Done 1800 / 5820
Done 2000 / 5820
Done 2200 / 5820
Done 2400 / 5820
Done 2600 / 5820
Done 2800 / 5820
Done 3000 / 5820
Done 3200 / 5820
Done 3400 / 5820
Done 3600 / 5820
Done 3800 / 5820
Done 4000 / 5820
Done 4200 / 5820
Done 4400 / 5820
Done 4600 / 5820
Done 4800 / 5820
Done 5000 / 5820
Done 5200 / 5820
Done 5400 / 5820
Done 5600 / 5820
Done 5800 / 5820
Done 5820 / 5820
(5820, 7)


In [8]:
quality_cols = ["contigs", "genome_length", "checkm_completeness", "checkm_contamination"]
for col in quality_cols:
    meta[col] = pd.to_numeric(meta[col], errors="coerce")

merged = labels.merge(meta, on="genome_id", how="left")
print(merged.shape)

(5820, 9)


In [9]:
good_rating   = merged["genome_quality"] == "Good"
few_contigs   = merged["contigs"] <= 300
right_size    = merged["genome_length"].between(5_000_000, 6_500_000)
complete      = merged["checkm_completeness"] >= 95
clean_genome  = merged["checkm_contamination"] <= 5

for name, rule in [("Good rating", good_rating), ("Contigs <= 300", few_contigs),
                   ("Length 5-6.5 Mb", right_size), ("Completeness >= 95", complete),
                   ("Contamination <= 5", clean_genome)]:
    print(f"{name}: {rule.sum()} pass")

passes_all = good_rating & few_contigs & right_size & complete & clean_genome
print("Pass all rules:", passes_all.sum())

Good rating: 5753 pass
Contigs <= 300: 5506 pass
Length 5-6.5 Mb: 5782 pass
Completeness >= 95: 5809 pass
Contamination <= 5: 5723 pass
Pass all rules: 5480


In [11]:
good = merged[passes_all]
subset = good.groupby("label").sample(n=500, random_state=42)
subset.to_csv(f"{PROJECT_DIR}/genomes_to_scan.csv", index=False)
print("Saved!")

Saved!


In [12]:
import shutil
import subprocess

RESULTS_DIR = f"{PROJECT_DIR}/amrfinder_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

SEQ_URL = "https://www.bv-brc.org/api/genome_sequence/"

to_scan = pd.read_csv(f"{PROJECT_DIR}/genomes_to_scan.csv", dtype={"genome_id": str})
to_scan = to_scan.groupby("label").sample(n=100, random_state=42)
to_scan = to_scan.sample(frac=1, random_state=42)

to_scan.to_csv(f"{PROJECT_DIR}/scan_list_200.csv", index=False)
print(to_scan["label"].value_counts())

already_done = len(os.listdir(RESULTS_DIR))
print(f"{len(to_scan)} genomes in the list, {already_done} already scanned")

label
Resistant      100
Susceptible    100
Name: count, dtype: int64
200 genomes in the list, 178 already scanned


In [13]:
for i, gid in enumerate(to_scan["genome_id"], start=1):
    out_file = f"{RESULTS_DIR}/{gid}.tsv"
    if os.path.exists(out_file):
        continue

    try:
        query = f"eq(genome_id,{gid})&limit(10000)"
        r = requests.get(SEQ_URL + "?" + query,
                         headers={"Accept": "application/dna+fasta"}, timeout=120)
        r.raise_for_status()
        with open("genome.fasta", "w") as f:
            f.write(r.text)

        subprocess.run(["amrfinder", "-n", "genome.fasta",
                        "--organism", "Klebsiella_pneumoniae",
                        "--name", gid, "--threads", "2",
                        "-o", "result.tsv"],
                       check=True, capture_output=True)

        shutil.copy("result.tsv", out_file)
        print(f"{i}: {gid} done")

    except Exception as e:
        print(f"{i}: {gid} FAILED: {e}")

    finally:
        if os.path.exists("genome.fasta"):
            os.remove("genome.fasta")

179: 573.66764 done
180: 573.67360 done
181: 573.21589 done
182: 573.13552 done
183: 573.67038 done
184: 573.66834 done
185: 573.23198 done
186: 573.26183 done
187: 573.67244 done
188: 573.24295 done
189: 573.14778 done
190: 573.5777 done
191: 573.21822 done
192: 573.26122 done
193: 573.13812 done
194: 573.67069 done
195: 573.21267 done
196: 573.20788 done
197: 573.67248 done
198: 573.15364 done
199: 573.46096 done
200: 573.67311 done
